# Interpret sequences with SAE features
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rotskoff-group/idiom/blob/v1/cookbook/notebooks/03_interpret_sae_features.ipynb)

Inspect feature rankings, residue traces, peak-centered logos, and feature presence across your sequences. You can reopen a saved feature dataset without loading model weights.

This notebook runs independently. Select **Runtime → Change runtime type → GPU** in Colab.
First use downloads model weights. A GPU is recommended; CPU inference is supported but slower. Reduce sample counts and batch size for a first run.
The setup installs the `v1` release when IDiom is absent. If using an older installation,
upgrade to that release and restart the kernel. No adjacent helper files are required.

In [ ]:
import importlib.util
import subprocess
import sys
if importlib.util.find_spec("idiom") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "idiom[cookbook] @ git+https://github.com/rotskoff-group/idiom.git@v1"])
if importlib.util.find_spec("pandas") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas>=2"])

import json
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from idiom import IDiom, IDiomSAE
from idiom.data.records import Record
from idiom.utils.notebook_helpers import (
    load_inputs, idr_sequence, isolated, check_context, summaries, write_fasta,
    save_run, sequence_metrics, nearest_reference, split_records, example_file,
)
print("Python:", sys.version.split()[0])
started = time.perf_counter()

## Inputs and settings
Run top to bottom. Upload a FASTA using Colab's Files pane and set its path below, or
leave the demo input unchanged. `INPUT_MODE="idr"` accepts ordinary headers for isolated
IDRs; `"annotated"` requires full-protein headers ending in `_IDR_x-y` (1-based inclusive).
Python coordinates are 0-based, end-exclusive. These workflows do not predict IDR boundaries.

For persistent outputs, optionally mount Drive in your own cell with
`from google.colab import drive; drive.mount("/content/drive")`, then set `OUT_DIR` there.
Use a new output directory for each experiment. Rejected records are reported in an audit.

In [ ]:
INPUT_FASTA = None
INPUT_MODE = "idr"
MAX_RECORDS = 16
SAE_ID = "jxliu2/idiomsae-300M-L18-k32"
DEVICE = "auto"
BATCH_SIZE = 2
FEATURE_IDS = None # None selects the three strongest active features
FEATURE_DIR = None # Or an existing dataset directory, e.g. "enrichment_outputs/fd_positive"
SEQUENCE_ROWS = [0, 1, 2]
N_WINDOWS = 20
HALF_WIDTH = 7
OUT_DIR = Path("inspection_outputs")

## Load or encode sequences
The demo consists of six illustrative sequences. The released SAE reads isolated IDRs using
its frozen 300M host model. A saved dataset preserves FIM positions, not original protein coordinates.
When reopening a dataset, its stored sequences define the analysis; INPUT_FASTA is ignored.

In [ ]:
from idiom.sae.features import FeatureDataset, AMINO_ACIDS, feature_windows, logo_data
OUT_DIR.mkdir(parents=True, exist_ok=True)
if FEATURE_DIR is None:
    records, audit = load_inputs(INPUT_FASTA, INPUT_MODE, MAX_RECORDS)
    audit.to_csv(OUT_DIR / "input_audit.csv", index=False)
    display(audit)
    if not records:
        raise ValueError("No accepted sequences; review the audit.")
    sae = IDiomSAE.from_pretrained(SAE_ID, device=DEVICE)
    check_context(records, sae.model.cfg.max_seq_len, include_flanks=sae.fim_mode == "prompted")
    FEATURE_DIR = sae.build_feature_dataset(records, OUT_DIR / "features", batch_size=BATCH_SIZE)
    index = summaries(records, audit)
    index.insert(0, "dataset_sequence", range(len(records)))
    index.to_csv(OUT_DIR / "sequence_index.csv", index=False)
dataset = FeatureDataset(FEATURE_DIR)
maximum, total, count = dataset.feature_ranking()
ranking = pd.DataFrame(dict(feature_id=range(dataset.num_latents), maximum=maximum,
                            total=total, firing_residues=count))
ranking = ranking.sort_values("maximum", ascending=False, kind="stable")
ranking.to_csv(OUT_DIR / "feature_ranking.csv", index=False)
feature_ids = ranking.loc[ranking.maximum > 0, "feature_id"].head(3).tolist() if FEATURE_IDS is None else FEATURE_IDS
display(ranking.head(10))

## Locate activations
Coordinates below index the stored FIM string (zero-based, excluding START). Marker characters
are not residue rows. Rankings and activations describe this dataset; they are not functional annotations.

In [ ]:
trace_rows = []
for feature in feature_ids:
    fig, ax = plt.subplots(figsize=(8, 2.5), constrained_layout=True)
    for seq_id in SEQUENCE_ROWS:
        if not 0 <= seq_id < dataset.n_seqs:
            continue
        positions, values = dataset.trace(seq_id, feature)
        ax.plot(positions, values, label=f"Sequence {seq_id}")
        trace_rows.extend(dict(sequence_id=seq_id, feature_id=feature, fim_position=int(p),
                               residue=dataset.sequence(seq_id)[p], activation=float(v))
                          for p, v in zip(positions, values))
    ax.set(title=f"Feature {feature}", xlabel="FIM position", ylabel="Activation")
    ax.legend()
    fig.savefig(OUT_DIR / f"feature_{feature}_trace.png", dpi=160)
    plt.show()
pd.DataFrame(trace_rows, columns=["sequence_id", "feature_id", "fim_position", "residue", "activation"]).to_csv(
    OUT_DIR / "residue_traces.csv", index=False)

## Inspect peak-centered windows
One window per sequence, ranked by peak activation. The peak stays centered; missing positions
at sequence ends remain missing rather than shifting the window. Letters show information relative
to a uniform amino-acid background, not proof of a biological motif. Inspect the exported windows too.

In [ ]:
import logomaker
window_rows = []
for feature in feature_ids:
    data = logo_data(dataset, feature, n=N_WINDOWS, half_width=HALF_WIDTH)
    fig, ax = plt.subplots(figsize=(7, 2), constrained_layout=True)
    if data["windows"]:
        logomaker.Logo(pd.DataFrame(data["information"], columns=list(AMINO_ACIDS)), ax=ax, color_scheme="chemistry")
        profile = data["mean_activation"]
        profile = profile / max(float(profile.max()), 1e-12)
        for i, value in enumerate(profile):
            ax.axvspan(i - 0.5, i + 0.5, color="orange", alpha=float(value) * 0.4, zorder=0)
    ax.axvline(HALF_WIDTH, color="black", linestyle="--", linewidth=0.5)
    ax.set_xticks(range(len(data["offsets"])), data["offsets"])
    ax.set(title=f"F{feature}: {len(data['windows'])} windows", xlabel="Offset from peak", ylabel="Bits")
    fig.savefig(OUT_DIR / f"feature_{feature}_logo.png", dpi=160)
    plt.show()
    np.savez(OUT_DIR / f"feature_{feature}_logo.npz", counts=data["counts"],
             mean_activation=data["mean_activation"], offsets=data["offsets"])
    window_rows.extend(dict(feature_id=feature, sequence_id=w.sequence_id,
                            peak_position=w.peak_position, window=w.residues) for w in data["windows"])
pd.DataFrame(window_rows, columns=["feature_id", "sequence_id", "peak_position", "window"]).to_csv(
    OUT_DIR / "feature_windows.csv", index=False)

## Compare sequences
The firing fraction measures encoded residues with positive activation. Presence means at least one positive firing.

In [ ]:
if feature_ids:
    fractions = np.stack([dataset.feature_stats(f)[2] for f in feature_ids], axis=1)
    pd.DataFrame(fractions, columns=feature_ids).to_csv(OUT_DIR / "firing_fractions.csv", index_label="sequence_id")
    fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
    im = ax.imshow(fractions[:30], aspect="auto", vmin=0, vmax=1, cmap="Oranges")
    ax.set_xticks(range(len(feature_ids)), feature_ids)
    ax.set(xlabel="Feature", ylabel="Sequence index")
    fig.colorbar(im, ax=ax, label="Firing fraction")
    fig.savefig(OUT_DIR / "feature_comparison.png", dpi=160)
    plt.show()
save_run(OUT_DIR, dict(feature_dir=str(FEATURE_DIR), sae=dataset.provenance, features=feature_ids),
         elapsed=time.perf_counter() - started)

## Save and continue
Use [signature discovery](04_discover_feature_signature.ipynb) for a positive/background comparison.

In [ ]:
import shutil
archive = shutil.make_archive(str(OUT_DIR.resolve()), "zip", OUT_DIR)
print("Results:", OUT_DIR.resolve(), "\nDownload:", archive)
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(archive)